# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">7주차 · 추천 파이프라인 — 추천 만드는 일을 명령 한 줄로 자동으로 하기</mark>

## 한눈에 보기

| 질문 | 답 |
|---|---|
| **오늘 무엇을 하나요?** | 6주차까지 노트북 셀을 하나씩 실행해서 만들던 추천을, 추천을 만드는 다섯 단계로 나누어 `recommend_batch.py` 파일 하나에 순서대로 적어 두고 **명령 한 줄**로 모두 실행합니다 |
| **그 결과로 무엇이 생기나요?** | 투자자 300명의 추천 10개씩이 들어 있는 **CSV 파일**(`recommendations.csv`)이 만들어집니다 |
| **왜 하나요?** | 고객에게 추천을 보여 주려면 추천을 날마다 새로 만들어 **파일로 저장해야** 합니다. 사람이 매일 셀을 하나씩 실행해서 파일을 만들기는 어렵습니다 |
| **추천 내용이 바뀌나요?** | **추천 방법**(행렬분해 + 위험도 상한)과 점수(Recall@10 = **0.3899**)는 6주차와 같습니다. 다만 파일은 가장 최근 기록까지 써서 만들기 때문에, 6주차 방식으로 만든 추천과 일부 종목이 다릅니다(7.3 에서 확인). Recall@10 은 다음 달 실제로 담은 종목 가운데 추천 10개 안에 든 비율입니다(1주차). **만드는 방법**이 바뀝니다 |

---

## 왜 필요한가 — 예시 상황

증권사 앱이 투자자 300명에게 **오늘의 추천 종목 10개**를 보여 주려 한다고 해 봅시다. 6주차까지의 방식(셀을 하나씩 실행)으로 하면 다음 문제가 생깁니다.

| 문제 | 실제로 생기는 일 |
|---|---|
| 날마다 해야 한다 | 상호작용 기록(`interactions` — 투자자가 종목을 조회 · 관심 · 구매한 기록)이 날마다 추가되므로 추천도 날마다 새로 만들어야 합니다. 사람이 매일 셀을 실행할 수는 없습니다 |
| 실수가 생긴다 | 셀을 빠뜨리거나 순서를 바꾸면 틀린 추천이 나옵니다. 추천이 잘못되어도 확인하기 어렵습니다 |
| 틀린 데이터를 확인하지 않는다 | 상호작용 기록에 오류가 있어도 확인하는 단계가 없어서, 오류가 있는 채로 학습합니다. 금융에서는 투자 성향에 맞지 않는 추천이 고객에게 가면 안 됩니다 |
| 결과가 화면에만 있다 | 노트북 화면에 나온 추천은 앱이 읽을 수 없습니다. 앱이 읽으려면 파일로 저장되어 있어야 합니다 |

**오늘 해결하는 방법**
- 실행 순서를 파일 하나에 적어 둡니다
- 시작하기 전에 데이터를 검사하고, 끝나기 전에 결과를 검사합니다
- 검사를 통과하면 파일로 저장합니다
- 데이터 점검부터 저장까지를 명령 한 줄로 실행합니다

---

## 오늘 만들어지는 파일 — 미리 보기

명령 한 줄을 실행하면 아래와 같은 파일이 만들어집니다 (앞 3줄).

| user_id | rank | item_id | item_name | risk_level | persona | score |
|---|---|---|---|---|---|---|
| U0001 | 1 | I002 | TIGER 미국배당다우존스 | 2 | 안정형 | 0.6211 |
| U0001 | 2 | I043 | Vanguard FTSE Developed Markets ETF | 3 | 안정형 | 0.4705 |
| U0001 | 3 | I014 | KT | 2 | 안정형 | 0.3974 |

- 한 줄은 **투자자 한 명에게 추천하는 종목 한 개**입니다.
- `rank` 는 **추천 순위**입니다. 1 은 그 투자자에게 **가장 먼저 추천하는 종목**이고, 10 은 열 번째로 추천하는 종목입니다.
  - 순위는 `score`(추천 점수)가 높은 순서입니다. 위 표에서 U0001 의 1등은 점수 0.6211 인 TIGER 미국배당다우존스이고, 2등은 점수 0.4705 인 Vanguard FTSE Developed Markets ETF 입니다.
- 열은 일곱 개이고, 각 열의 뜻은 다음과 같습니다.
  - `user_id` — 투자자 번호입니다 (U0001 \~ U0300, 300명).
  - `rank` — 추천 순위입니다 (위에서 설명).
  - `item_id` — 종목 번호입니다 (I001 \~ I100). 종목마다 번호가 하나씩 정해져 있습니다.
  - `item_name` — 종목 이름입니다.
  - `risk_level` — 그 종목의 위험도입니다. 1 이 가장 낮고 5 가 가장 높습니다.
  - `persona` — 그 투자자의 투자 성향입니다 (안정형 · 성장형 · 공격형). 성향마다 추천할 수 있는 위험도 상한이 3 · 4 · 5 로 정해져 있습니다.
  - `score` — 추천 점수입니다. 행렬분해로 계산한 값이고, 높을수록 그 투자자에게 더 맞는다고 본 종목입니다. 이 점수 순서가 `rank` 가 됩니다.
- 투자자 300명 × 10개 = **3,000줄**입니다.
- 다음 주(8주차)에 FastAPI 가 이 파일을 읽습니다. 앱이 투자자 번호를 보내면, 그 투자자의 추천 10개를 응답합니다.

### 명령 한 줄이 하는 일 — 파이프라인 5단계

**파이프라인(pipeline)** 은 데이터 점검 · 학습 · 추천 계산 · 결과 검사 · 저장처럼 여러 작업을 **정해진 순서로 한 번에 실행하는 방식**입니다. 각 작업의 결과를 다음 작업에서 사용하고, 문제가 발견되면 그 뒤의 작업은 진행하지 않고 실행을 중단합니다.

| 단계 | 하는 일 | 왜 필요한가 | 오늘 어디서 |
|---|---|---|---|
| ① 데이터 점검 | 상호작용 기록에 빈 값 · 중복 · 목록에 없는 번호가 없는지 검사한다 | 오류가 있는 데이터로 학습하면 틀린 결과가 나올 수 있다 | 7.1 |
| ② 학습 | 상호작용 기록 전체로 행렬분해 모델을 만든다 | 추천 점수를 계산하려면 모델이 필요하다 | 7.2 · 7.3 |
| ③ 일괄 추천 | 300명 각각에게 추천 10개를 한꺼번에 계산한다 (위험도 상한 포함. 투자 성향별로 추천할 수 있는 위험도의 최댓값이며 6주차에서 정했다) | 앱이 요청할 때마다 계산하지 않고, 미리 만들어 둔다 | 7.3 |
| ④ 규격 검사 | 결과 표가 정해 둔 파일 형식(열 이름 · 순서 · 개수)에 맞는지 검사한다 | 모양이 틀린 파일이 앱으로 넘어가지 않게 한다 | 7.3 · 7.4 |
| ⑤ 저장 | `output/recommendations.csv` 로 저장한다 | 다음 주 API 가 읽을 수 있게 파일로 남긴다 | 7.3 · 7.4 |

다섯 단계의 코드는 모두 **`recommend_batch.py` 파일 하나**에 적혀 있습니다. 이 노트북에서는 먼저 그 파일의 코드를 읽고, 이어서 실행한 다음, 만들어진 결과를 확인합니다.

---

### 6주차까지와 달라지는 것

| | 6주차까지 | 오늘 |
|---|---|---|
| 추천을 만드는 방법 | 셀을 하나씩 직접 실행 | 명령 한 줄 |
| 실행하는 쪽 | 사람 | 컴퓨터 |
| 결과 | 화면에만 나타남 | **CSV 파일**로 저장됨 |
| 추천 방법 · 점수 | 행렬분해 + 위험도 상한 · Recall@10 0.3899 | 같음 |
| U0001 의 1등 | TIGER 미국배당다우존스 | 같음 (10개 중 8개가 같고 2개는 다름) |

---

### 오늘의 구성 — 50분 (PPT + 노트북)

| 시간 | 어디서 | 하는 일 |
|---|---|---|
| 개념 30분 | PPT | 파이프라인 · 스크립트 · 데이터 점검 · 학습과 서빙 · 일괄 추천 · output schema · 재현성 |
| 실습 20분 | **이 노트북** | 모두 실행 → 7.1 데이터 점검 → 7.2 스크립트 읽기 → 7.3 실행 → 7.4 결과 파일 → 🗣 함께 보기 |

### 이 노트북 읽는 법 — 🎯 와 📖

- **🎯 수업** — 수업 시간에 다루는 절입니다
  - 셀이 있으면 실행하고 **결과를 함께 읽습니다**
- **📖 혼자 읽기** — 수업에서는 건너뛰고 **복습할 때 읽는** 절입니다
  - 7.5 · 7.6 · 7.7 · 7.8 · 7.9 가 여기에 속합니다

⏱ **실습을 시작하면 먼저 메뉴의 `런타임 → 모두 실행` 을 눌러 두세요.** 전체 실행에 10초쯤 걸립니다. 그동안 위에서부터 결과를 읽어 내려가면 됩니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/07_batch_pipeline.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것** — 모두 **파일**입니다. 셀 안에 적혀 있지 않고, 아래 셀이 내려받는 폴더 안에 들어 있습니다.

| 무엇 | 어디에 | 쓰임 |
|---|---|---|
| `items.csv` · `users.csv` · `interactions.csv` | **`data/` 폴더** | `items`(종목) 100개 · `users`(투자자) 300명 · `interactions`(상호작용 — 조회 · 관심 · 구매 기록) 8,086건 |
| `recsys.py` | `notebooks/` 폴더 | 데이터 읽기 · 시간순 분할 · 채점 · 사용자-아이템 행렬 |
| `recommend_batch.py` | `notebooks/` 폴더 | **오늘의 산출물** — 데이터 점검부터 저장까지 실행 한 번으로 하는 스크립트 |

- Colab 에서는 아래 셀이 저장소를 `/content/recsys` 에 내려받습니다. 그러면 데이터는 `/content/recsys/data`, 파이썬 파일은 `/content/recsys/notebooks` 에 생깁니다.
- 왼쪽의 📁 아이콘에서 `recsys` 폴더를 열면 파일을 직접 볼 수 있습니다.

**이 노트북이 만드는 것**

| 무엇 | 어디에 |
|---|---|
| 300명 × 추천 10개 결과 파일 | `output/recommendations.csv` |

⚠ **지난주 노트북을 먼저 돌릴 필요가 없습니다.** 이 노트북 하나로 끝납니다.

⚠ **따로 설치할 것이 없습니다.** 오늘 쓰는 도구는 Colab 에 처음부터 들어 있습니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 새 환경에서 시작하므로 데이터·recsys.py 가 없다. 내 컴퓨터에서 실행할 때는 자료를 내려받는 이 단계가 필요 없다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 일곱 개</mark>

여기서 모두 외울 필요는 없습니다. 괄호 안의 절에서 예와 함께 다시 설명합니다. 지금은 이름만 읽고, 7.1 부터 하나씩 직접 확인합니다.

1. **파이프라인 (pipeline)** — 여러 단계를 순서대로 이어서 **한 번에 실행하는 것** (맨 위 표)
   - 오늘의 파이프라인은 데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장입니다

2. **스크립트 (script)** — 파이썬 코드를 `.py` 파일에 저장해 두고, 명령 한 줄로 **처음부터 끝까지 한 번에 실행하는 것** (7.2)
   - 노트북(`.ipynb`)은 셀을 하나씩 실행하지만, 스크립트는 파일 전체가 한 번에 실행됩니다
   - 오늘의 스크립트는 `recommend_batch.py` 입니다

3. **데이터 점검 (data validation)** — 학습하기 **전에** 입력 데이터에 오류가 없는지 검사하는 일 (7.1)
   - 빈 값 · 중복 · 목록에 없는 번호 같은 것을 셉니다
   - 노션 계획표의 「데이터 정리」에 해당합니다

4. **학습 (training)** 과 **서빙 (serving)** (7.2 · 7.6)
   - 학습 — 상호작용 기록으로 모델을 만드는 일. 시간이 오래 걸리고, 하루에 한 번처럼 **가끔** 합니다
   - 서빙 — 미리 만들어 둔 학습 결과로 고객에게 추천을 제공하는 일. 모델을 새로 만들지 않아서 빠르고, 고객이 앱을 열 때마다 **자주** 합니다
   - 둘을 나누면 고객이 앱을 열 때마다 학습을 다시 하지 않아도 됩니다

5. **일괄 추천 (batch recommendation)** — 모든 고객의 추천을 **한꺼번에 미리** 계산해 두는 방식 (7.3)
   - 반대로 고객이 요청할 때 그 자리에서 계산하는 방식은 **실시간 추천(real-time recommendation)** 입니다
   - 오늘은 모든 투자자의 추천을 미리 계산합니다. 8주차의 API 는 요청이 오면 미리 만든 결과 파일에서 해당 투자자의 추천을 읽어 응답합니다

6. **output schema (결과 파일 규격)** 와 **규격 검사** (7.4)
   - 규격 — 결과 파일의 **열 이름 · 순서 · 개수**를 정해 둔 약속
   - 규격 검사 — 결과가 그 약속을 지켰는지 코드가 자동으로 확인하는 것
   - 데이터 점검은 **들어오는** 데이터의 검사, 규격 검사는 **나가는** 파일의 검사입니다

7. **재현성 (reproducibility)** — 같은 입력으로 다시 실행하면 **같은 결과**가 나오는 성질 (7.7)
   - `random_state=42` 처럼 무작위 요소를 고정해서, 같은 입력에서 같은 결과가 나오게 합니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 준비 — 데이터 읽기와 시간순 분할</mark>

**여기서 하는 일** — 데이터를 읽고, 3주차부터 쓰던 **시간순 분할**로 나눕니다. 그리고 오늘의 스크립트 `recommend_batch.py` 를 불러옵니다.

- 시간순 분할은 7.5(채점)와 7.6(시간 재기)에서 씁니다. 7.5 에서는 이 분할로 채점해야 6주차 **0.3899** 와 비교할 수 있습니다.
- `import recommend_batch` — 스크립트 파일 안의 함수를 노트북에서도 사용할 수 있게 불러옵니다.

In [2]:
# 이 셀에서 하는 일 — 도구를 불러오고, 데이터를 읽고, 시간순으로 나눈다
# 왜 하나 — 7.1 에서 데이터를 점검하고, 7.5 · 7.6 에서 6주차와 같은 기준으로 채점하고 시간을 재려면 필요하다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py · recommend_batch.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. pd 라고 부른다
import recsys                               # 이 수업용 도구 모음 — 읽기 · 분할 · 채점
import recommend_batch                      # 오늘의 스크립트 — 안에 있는 함수를 노트북에서도 사용할 수 있다

items, users, interactions = recsys.load()                  # items(종목) · users(투자자) · interactions(상호작용) 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 3주차부터 쓰던 시간순 분할

print(f"items(종목) {len(items)}개 · users(투자자) {len(users)}명 · interactions(상호작용) {len(interactions):,}건")
print(f"채점용 — 학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건")

items(종목) 100개 · users(투자자) 300명 · interactions(상호작용) 8,086건
채점용 — 학습 구간 7,095건 · 채점 구간 991건


**결과 읽는 법**

**결론: 이 수업의 데이터는 종목 100개 · 투자자 300명 · 상호작용 8,086건입니다. 이 중 마지막 1개월(991건)은 채점에만 씁니다.**
- 첫 줄 — 오늘 쓰는 표 세 개의 크기입니다
- 둘째 줄 — 채점용 분할입니다. **학습 구간 7,095건**으로 학습하고 **채점 구간 991건**으로 채점합니다 (6주차와 같음)
- 스크립트가 만드는 **서비스용** 추천은 나누지 않고 8,086건 **전체**로 학습합니다 (7.3)

**데이터 파일 세 개 살펴보기** — `data/` 폴더의 CSV 파일을 표로 읽은 것이 `items`(종목) · `users`(투자자) · `interactions`(상호작용) 입니다. 앞 3줄씩 봅니다.

- `head(3)` 는 표의 **앞 3줄**만 보여 줍니다.


In [3]:
# 이 셀에서 하는 일 — data 폴더의 세 파일(items · users · interactions)을 앞 3줄씩 본다
# 왜 하나 — 오늘 쓰는 데이터가 어떤 열로 되어 있는지 눈으로 확인한다

print("① items.csv (종목) — 100개")
display(items.head(3))
print("② users.csv (투자자) — 300명")
display(users.head(3))
print("③ interactions.csv (상호작용: 조회 · 관심 · 구매 기록) — 8,086건")
display(interactions.head(3))

① items.csv (종목) — 100개


,item_id,name,ticker,market,asset_class,sector,risk_level,theme
0,I001,ACE 미국배당다우존스,402970,KR,ETF,배당,2,배당
1,I002,TIGER 미국배당다우존스,458730,KR,ETF,배당,2,배당
2,I003,KODEX 미국배당다우존스,489250,KR,ETF,배당,2,배당


② users.csv (투자자) — 300명


,user_id,persona,subcluster,risk_appetite,joined_at,is_newcomer
0,U0001,안정형,안정형-1,1.8,2025-11-15,False
1,U0002,성장형,성장형-3,3.6,2026-01-15,False
2,U0003,공격형,공격형-4,4.7,2026-01-30,False


③ interactions.csv (상호작용: 조회 · 관심 · 구매 기록) — 8,086건


,user_id,item_id,ts,event
0,U0245,I043,2025-08-31 10:00:00,view
1,U0055,I032,2025-09-03 12:00:00,like
2,U0055,I008,2025-09-04 11:00:00,view


**결과 읽는 법**

**결론: 세 파일은 번호(`item_id` · `user_id`)로 연결됩니다. 추천에 쓰는 것은 상호작용이고, 종목 표와 투자자 표는 위험도 · 투자 성향을 알려 줍니다.**
- **`items.csv`** — 종목 한 개가 한 줄
  - `item_id` 종목 번호 · `name` 종목 이름 · `risk_level` 위험도(1 \~ 5) · `sector` · `theme` 는 종목의 분류
  - 결과 파일의 `item_name` 은 이 표의 `name`
- **`users.csv`** — 투자자 한 명이 한 줄
  - `persona` 투자 성향이 위험도 상한(안정형 3 · 성장형 4 · 공격형 5)을 정함
  - `risk_appetite` 는 위험 선호 점수(1.8 \~ 4.7)
- **`interactions.csv`** — 투자자가 종목에 한 행동 하나가 한 줄
  - `ts` 시각 · `event` 는 `view`(조회) · `like`(관심) · `buy`(구매)
  - 이 수업은 세 종류를 모두 「그 종목을 담았다」는 기록으로 셉니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.1 데이터 점검 — 학습하기 전에 입력 데이터를 검사한다</mark>

**파이프라인에서의 위치:** ① 데이터 점검 — 추천을 만들기 전에 상호작용 기록(조회 · 관심 · 구매 기록)에 오류가 없는지 확인합니다.


**7.1 에서 하는 일**
- `interactions`(상호작용) · `items`(종목) · `users`(투자자) 세 표에서 문제가 있는 줄이 몇 개인지 **여섯 가지**로 셉니다. 모두 0 이면 통과입니다.
- 그다음 문제가 있는 데이터를 **일부러 만들어서**, 점검 결과가 어떻게 달라지는지 확인합니다.

**왜 하는가**
- 지금까지는 데이터에 문제가 없다고 가정하고 사용했습니다. 날마다 자동으로 실행하면 사람이 매번 데이터를 확인하기 어렵습니다.
- 문제가 있는 데이터로 학습하면 **오류 메시지 없이 틀린 결과**가 나올 수 있습니다.
  - 예 — 같은 상호작용 기록이 두 번 포함되면 그 종목의 상호작용 횟수가 실제보다 많게 계산됩니다. 상호작용 횟수를 세어 쓰는 계산(예 · 인기 순위)은 오류 메시지 없이 틀어집니다.
  - 오늘의 행렬분해는 담았으면 1, 아니면 0으로만 쓰기 때문에 이 10줄만으로는 추천이 바뀌지 않았습니다. 그래도 입력 데이터를 믿을 수 있는지는 학습 전에 확인해야 합니다.
- 그래서 학습하기 전에 먼저 검사하고, 문제가 있으면 **실행을 중단**합니다.

**여섯 가지 검사**

| 검사 | 확인하는 것 | 문제가 되는 이유 |
|---|---|---|
| 빈 값 | 값이 비어 있는 곳 | 계산이 안 되거나 틀린 결과가 나옴 |
| 똑같은 상호작용 기록 (중복) | 모든 열의 값이 같은 줄이 두 번 이상 있는가 | 그 종목의 상호작용 횟수가 실제보다 많게 계산됨 |
| 종목 목록에 없는 종목 번호 | 상호작용 기록의 종목 번호가 종목 표에 있는가 | 그 기록이 오류 메시지 없이 학습에서 빠짐 (종목 이름과 위험도를 알 수 없음) |
| 투자자 목록에 없는 투자자 번호 | 상호작용 기록의 투자자 번호가 투자자 표에 있는가 | 오류 없이 학습에 들어가지만, 투자 성향을 알 수 없어 추천 파일에는 나오지 않음 |
| 세 가지가 아닌 투자성향 | 안정형 · 성장형 · 공격형 외의 값이 있는가 | 위험도 상한을 정할 수 없음 |
| 1 \~ 5 를 벗어난 위험도 | 위험도가 1 에서 5 사이인가 | 위험도 상한과 비교한 결과가 틀림 |

**순서**
1. 스크립트의 `데이터_점검` 함수에 세 표를 넘깁니다
2. 함수가 여섯 가지 문제의 개수를 셉니다
3. 모두 0 이면 통과, 하나라도 0 이 아니면 학습을 시작하지 않고 실행을 중단합니다

In [4]:
# 이 셀에서 하는 일 — 스크립트의 데이터 점검 함수를 불러 여섯 가지 검사 결과를 표로 본다
# 왜 하나 — 학습 전에 데이터에 오류가 있는지 확인한다. 여섯 가지 검사의 문제 개수가 모두 0 이어야 통과다

점검 = recommend_batch.데이터_점검(items, users, interactions)   # {검사 이름: 문제 개수}
점검_표 = pd.DataFrame({"검사": list(점검.keys()), "문제 개수": list(점검.values())})
print(점검_표.to_string(index=False))
print(f"\n상호작용 기록 날짜 — {interactions['ts'].min().date()} 부터 {interactions['ts'].max().date()} 까지")
print("→ 통과" if sum(점검.values()) == 0 else "→ 실패 — 학습하지 않고 실행을 중단해야 한다")

               검사  문제 개수
              빈 값      0
      똑같은 상호작용 기록      0
  종목 목록에 없는 종목 번호      0
투자자 목록에 없는 투자자 번호      0
    세 가지가 아닌 투자성향      0
  1 ~ 5 를 벗어난 위험도      0

상호작용 기록 날짜 — 2025-08-31 부터 2026-08-30 까지
→ 통과


**결과 읽는 법**

**결론: 여섯 가지 검사 모두 문제가 0건이라 통과입니다. 이 데이터로 학습해도 됩니다.**
- 표의 한 줄이 검사 하나이고, 「문제 개수」가 모두 0 입니다
- 날짜는 2025-08-31 \~ 2026-08-30 (1년치)이고 미래 날짜는 없습니다
- 수업용으로 만든 데이터라 문제가 없습니다. 실제 데이터에서는 0 이 아닌 숫자가 자주 나옵니다
  - 그래서 바로 아래에서 문제를 **일부러 넣어** 점검이 찾아내는지 확인합니다

**검사 여섯 개가 문제 개수를 세는 방법**

`데이터_점검` 안의 여섯 검사는 모두 **같은 방법**입니다. 문제가 있는 줄을 `True` 로 표시하고, `sum()` 으로 `True` 의 개수를 셉니다. 0 이면 통과입니다.

| 검사 | 쓰는 pandas 함수 | 세는 방법 |
|---|---|---|
| 빈 값 | `isna()` | 값이 비어 있으면 `True`. 표 세 개의 개수를 더합니다 |
| 똑같은 상호작용 기록 | `duplicated()` | 모든 열의 값이 앞에 나온 줄과 같으면 `True` |
| 종목 목록에 없는 종목 번호 | `isin()` 과 `~` | 종목 표에 **있으면** `True` → `~` 로 뒤집어 **없는** 번호를 셉니다 |
| 투자자 목록에 없는 투자자 번호 | `isin()` 과 `~` | 투자자 표와 같은 방법으로 비교합니다 |
| 세 가지가 아닌 투자성향 | `isin()` 과 `~` | `안정형` · `성장형` · `공격형` 에 **없는** 값을 셉니다 |
| 1 \~ 5 를 벗어난 위험도 | `between(1, 5)` 와 `~` | 1 이상 5 이하이면 `True` → 뒤집어 벗어난 값을 셉니다 |

- `~` 는 `True` 와 `False` 를 서로 바꿉니다. `isin` 은 「있는가」를 묻기 때문에, 앞에 `~` 를 붙여 「없는 것」을 셉니다.

작은 예로 각 함수가 어떻게 `True` / `False` 를 만드는지 봅니다.

In [5]:
# 이 셀에서 하는 일 — 데이터 점검에 쓰는 pandas 함수 네 가지(isna · duplicated · isin · between)를 작은 예로 본다
# 왜 하나 — 여섯 가지 검사가 문제 개수를 어떻게 세는지 눈으로 확인한다

print("빈 값 — isna()")
값들 = pd.Series([1, None, 3])                                    # 가운데 값이 비어 있다
print("  isna      :", list(값들.isna()), "→ 개수", int(값들.isna().sum()))

print("\n똑같은 상호작용 기록 — duplicated()")
작은_기록 = pd.DataFrame({"user_id": ["U1", "U2", "U2"], "item_id": ["I001", "I002", "I002"]})   # 2 번째와 3 번째 줄이 같다
print("  duplicated:", list(작은_기록.duplicated()), "→ 개수", int(작은_기록.duplicated().sum()))   # 앞에 나온 줄과 같은 줄만 True

print("\n목록에 없는 번호 — ~isin()  (종목 번호 · 투자자 번호 · 투자성향이 모두 같은 방법)")
번호들 = pd.Series(["I001", "I999"])                              # I999 는 종목 표에 없다
있음 = 번호들.isin(items["item_id"])                              # 종목 표에 있으면 True
print("  isin      :", list(있음), "→ ~ 로 뒤집기", list(~있음), "→ 개수", int((~있음).sum()))

print("\n범위를 벗어난 값 — ~between(1, 5)")
위험도들 = pd.Series([3, 6])                                      # 6 은 1 ~ 5 를 벗어난다
범위_안 = 위험도들.between(1, 5)                                  # 1 이상 5 이하이면 True
print("  between   :", list(범위_안), "→ ~ 로 뒤집기", list(~범위_안), "→ 개수", int((~범위_안).sum()))

빈 값 — isna()
  isna      : [False, True, False] → 개수 1

똑같은 상호작용 기록 — duplicated()
  duplicated: [False, False, True] → 개수 1

목록에 없는 번호 — ~isin()  (종목 번호 · 투자자 번호 · 투자성향이 모두 같은 방법)
  isin      : [True, False] → ~ 로 뒤집기 [False, True] → 개수 1

범위를 벗어난 값 — ~between(1, 5)
  between   : [True, False] → ~ 로 뒤집기 [False, True] → 개수 1


**결과 읽는 법**

**결론: 네 가지 함수 모두 「문제가 있는 줄 = `True`」로 표시하고, 그 개수를 세는 같은 방법입니다.**
- `isna()` — 비어 있는 값만 `True` → 개수 **1**
- `duplicated()` — 앞에 나온 줄과 같은 세 번째 줄만 `True` → 개수 **1**
- `isin()` 과 `~` — `I999` 가 종목 표에 없어서 `False`. `~` 로 뒤집으면 없는 번호가 `True` → 개수 **1**
- `between(1, 5)` 와 `~` — `6` 이 범위를 벗어나서 `False`. 뒤집으면 `True` → 개수 **1**
- 위의 `데이터_점검` 은 이 방법을 표 전체에 적용한 것이고, 우리 데이터는 문제가 없어서 모든 개수가 **0** 이었습니다

**문제가 있는 데이터를 일부러 넣어 보기 — 점검이 정말 문제를 찾아내는지 확인하는 테스트**

**왜 하나**
- 우리 데이터는 문제가 없어서 점검 결과가 계속 0 입니다. 그런데 0 만 봐서는 두 경우를 구분할 수 없습니다.
  - 데이터에 **정말 문제가 없어서** 0 인 경우
  - 점검이 **문제를 못 찾아서** 0 인 경우
- 그래서 **문제를 일부러 만들어 넣고**, 점검이 찾아내는지 확인합니다.

**방법**
1. 상호작용 기록의 **앞 10줄을 복사해서 맨 뒤에 한 번 더 붙입니다.** 똑같은 줄 10쌍이 생깁니다 (8,086건 → 8,096건).
2. 이 표를 같은 `데이터_점검` 함수에 넘깁니다.
3. 「똑같은 상호작용 기록」이 **10건**으로 나오면 점검이 제대로 동작하는 것입니다.

- 이 중복 10건은 **실제 데이터의 문제가 아니라 테스트용으로 만든 가짜 문제**입니다.
- 복사본에만 넣습니다. 원래 데이터 `interactions` 는 바뀌지 않으므로 다음 셀부터는 다시 0 건입니다.
- 결과에 「실패」가 나오는 것이 **정상**입니다. 점검이 문제를 찾았다는 뜻입니다.

In [6]:
# 이 셀에서 하는 일 — 상호작용 기록의 앞 10줄을 복사본에 한 번 더 붙여 일부러 문제를 만들고, 데이터 점검이 찾아내는지 본다
# 왜 하나 — 점검 결과 0 이 「문제가 없어서」인지 「점검이 못 찾아서」인지 구분하려고, 문제를 넣은 데이터로 점검을 시험한다. 원래 데이터는 바꾸지 않는다

문제_있는_기록 = pd.concat([interactions, interactions.head(10)], ignore_index=True)   # concat = 표를 위아래로 합친다. 앞 10줄을 한 번 더 넣고, 행 번호는 새로 매긴다
문제_있는_점검 = recommend_batch.데이터_점검(items, users, 문제_있는_기록)  # 같은 함수에 문제가 있는 표를 넘긴다

print(f"상호작용 기록 {len(interactions):,}건 → {len(문제_있는_기록):,}건")
print(f"똑같은 상호작용 기록 : {문제_있는_점검['똑같은 상호작용 기록']}건")
print("→ 통과" if sum(문제_있는_점검.values()) == 0 else "→ 실패 — 스크립트라면 여기서 실행을 중단하고 학습을 시작하지 않는다")

상호작용 기록 8,086건 → 8,096건
똑같은 상호작용 기록 : 10건
→ 실패 — 스크립트라면 여기서 실행을 중단하고 학습을 시작하지 않는다


**결과 읽는 법**

**결론: 일부러 넣은 중복 10건을 점검이 정확히 10건으로 찾아냈습니다. 점검이 제대로 동작한다는 확인이고, 「실패」는 점검이 문제를 찾았다는 정상 결과입니다.**
- 복사본의 상호작용 기록이 8,086건 → 8,096건 (앞 10줄을 한 번 더 붙임)
- 똑같은 상호작용 기록 **10건** 검출 — 앞 셀(원래 데이터)에서는 0건이었습니다
- 스크립트라면 이때 **학습을 시작하지 않고 멈춥니다**
  - 원인을 모르는 채 학습하면 틀린 추천 3,000개가 만들어질 수 있기 때문입니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.2 스크립트 읽기 — 다섯 단계가 어디에 있는지 본다</mark>

**파이프라인에서의 위치:** ① 에서 ⑤ 까지 — 5단계가 파일 안 어디에 적혀 있는지 읽습니다. 아직 실행하지는 않습니다.


**7.2 에서 하는 일** — 실행하기 **전에** `recommend_batch.py` 의 `main` 함수를 화면에 띄우고, 다섯 단계가 어디에 있는지 읽습니다.

**스크립트(script)란**
- 파이썬 코드를 `.py` 파일에 저장한 것입니다. 노트북(`.ipynb`)처럼 셀을 하나씩 실행하지 않고, **명령 한 줄로 파일 전체를 처음부터 끝까지** 실행합니다.

| | 노트북 (`.ipynb`) | 스크립트 (`.py`) |
|---|---|---|
| 실행하는 방법 | 셀을 하나씩 실행 | 명령 한 줄로 처음부터 끝까지 |
| 주로 쓰는 곳 | 실험 · 설명 · 결과 확인 | 반복해서 자동으로 실행 |

**`main` 함수란**
- 스크립트를 실행하면 **시작점이 되는 함수**입니다. 이 파일이 하는 일의 순서가 `main` 안에 적혀 있어서, `main` 만 읽어도 전체 흐름을 알 수 있습니다.

**읽는 순서** — `main` 안에서 아래 순서로 나옵니다.

| 단계 | `main` 안의 코드 | 하는 일 |
|---|---|---|
| ① 데이터 점검 | `데이터_점검(...)` | 하나라도 0 이 아니면 **실행을 중단** |
| ② 학습 | `학습하기(interactions, items)` | 상호작용 기록 **전체**로 모델을 만든다 |
| ③ 일괄 추천 | `일괄_추천(...)` | 300명 × 10개 = 3,000줄 표를 만든다 |
| ④ 규격 검사 | `규격_검사(...)` | 문제가 있으면 **저장하기 전에 실행을 중단** |
| ⑤ 저장 | `표.to_csv(...)` | CSV 파일로 저장한다 |

- **저장(⑤)은 검사(④) 뒤에 있습니다.** 검사를 통과하지 못한 결과는 파일로 저장되지 않습니다.
- 오늘 사용하는 모델은 6주차의 최종 추천(**행렬분해 + 위험도 상한**)입니다. 6주차에서 비교한 네 모델 중 점수가 가장 높았기 때문입니다.

In [7]:
# 이 셀에서 하는 일 — 스크립트의 main 함수를 글자 그대로 화면에 띄운다
# 왜 하나 — 실행하기 전에 다섯 단계가 어떤 순서로 있는지 읽는다

import inspect                                     # 함수의 코드를 글자 그대로 보여 주는 도구

print(inspect.getsource(recommend_batch.main))     # getsource = 함수의 코드를 글자 그대로 가져온다

def main():
    items, users, interactions = recsys.load()               # data 폴더의 CSV 세 개를 표로 읽는다

    # ① 데이터 점검 — 하나라도 문제가 있으면 여기서 멈춘다. 학습도 저장도 하지 않는다
    점검 = 데이터_점검(items, users, interactions)
    if sum(점검.values()) > 0:
        raise SystemExit(f"① 데이터 점검 실패 — {점검}")      # SystemExit = 스크립트를 끝낸다. 메시지가 화면에 나온다
    print(f"① 데이터 점검 — 여섯 가지 모두 0건 · 통과")

    # ② 학습 — 시간순으로 나누지 않고 전체를 쓴다. 채점할 미래가 없고 가장 최근 기록까지 반영해야 하기 때문이다
    묶음 = 학습하기(interactions, items)                    # 서비스용은 상호작용 기록 전체로 학습
    print(f"② 학습 — 상호작용 기록 {len(interactions):,}건 · 투자자 {len(묶음['사용자_목록'])}명")

    # ③ 일괄 추천 — 300명 모두의 추천을 표 하나로 만든다
    표 = 일괄_추천(묶음, users, items)
    print(f"③ 일괄 추천 — {len(표):,}줄")

    # ④ 규격 검사 — 어긴 것이 있으면 저장하지 않고 멈춘다 (앞의 5개만 화면에 보여 준다)
    문제 = 규격_검사(표, users, 묶음)
    if 문제:
        raise SystemExit("④ 규격 검사 실패 — " + " / ".join(문제[:5]))
    print("④ 규격 검사 — 문제 0개 · 통과")

    # ⑤ 저장 — ① 과 ④ 를 모두 통과했을 때만 여기에 온다
    출력_파일.parent.mkdir(exist_ok=True)                    # output 폴더가

**결과 읽는 법**

**결론: `main` 만 읽으면 ① \~ ⑤ 의 순서와 「어디서 멈추는지」를 알 수 있습니다.**
- 단계마다 `print(f"① …")` 줄이 하나씩 있어서, 실행하면 어디까지 진행되었는지 화면에 나옵니다
- 멈추는 곳은 두 군데입니다 (`raise SystemExit`)
  - `if sum(점검.values()) > 0:` — 데이터 점검 실패 → **학습 전**에 중단
  - `if 문제:` — 규격 검사 실패 → **저장 전**에 중단
- `학습하기(interactions, items)` 는 채점용 `train` 이 아니라 **전체 `interactions`** 로 학습합니다. 서비스용이기 때문이며 이유는 7.3 에서 설명합니다
- `main` 이 부르는 함수 **안**의 로직은 단계에 맞는 절에서 봅니다 — `데이터_점검` 7.1 · `추천하기` 7.3 · `규격_검사` 7.4 · `학습하기` 등 나머지 📖 7.8

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.3 스크립트 실행 — 명령 한 줄로 다섯 단계를 끝까지</mark>

**파이프라인에서의 위치:** ① 에서 ⑤ 까지 — 5단계를 명령 한 줄로 끝까지 실행합니다.


**7.3 에서 하는 일** — `!python recommend_batch.py` **한 줄**로 스크립트를 실행합니다. 셀을 하나씩 실행하지 않아도 데이터 점검부터 저장까지 끝납니다.

**명령을 읽는 법**
- 셀 맨 앞의 `!` — 이 줄은 파이썬 코드가 아니라 **컴퓨터에 내리는 명령**이라는 표시입니다 (Colab · 주피터에서만 씁니다)
- `python recommend_batch.py` — 파이썬으로 `recommend_batch.py` 파일을 실행하라는 뜻입니다

**왜 서비스용은 전체 기간으로 학습하나**

| | 채점용 (7.5) | 서비스용 (오늘의 스크립트) |
|---|---|---|
| 학습에 쓰는 기록 | 학습 구간 7,095건 | **전체 8,086건** |
| 목적 | 마지막 1개월의 기록을 학습에 쓰지 않고, 그 기록으로 추천 성능을 평가하기 (3주차) | 고객에게 제공할 추천 만들기 |
| 추천 성능을 평가할 수 있나 | 가능 — 학습에 쓰지 않은 마지막 1개월 기록과 비교 | 불가 — 결과를 확인할 미래 기록이 아직 없음 |

- 서비스용 추천은 **앞으로 고객에게 제공할** 것이라, 가장 최근 기록까지 모두 써야 합니다.
- 그래서 파일의 추천은 6주차 방식(학습 구간만 사용)으로 만든 추천과 일부 종목이 다릅니다. 300명 중 289명이 10개 중 1개 이상 다르고, 평균 7.45개가 같습니다(혼자 해 보기 2 의 답에서 확인합니다).

**실행하면 일어나는 일**
1. 컴퓨터가 새 파이썬을 시작해 `recommend_batch.py` 를 맨 위부터 읽습니다
2. 맨 아래 `if __name__ == "__main__": main()` 에서 `main` 이 실행됩니다
3. `main` 이 ① 에서 ⑤ 까지 차례로 실행하고, 단계마다 한 줄씩 출력합니다
4. 끝나면 `output/recommendations.csv` 파일이 만들어집니다

- 이 셀은 노트북 안의 변수(`items` · `train` 등)를 **쓰지 않습니다.** 스크립트가 데이터를 직접 읽습니다. 그래서 위의 데이터 읽기 셀(준비)이나 7.1 의 셀을 실행하지 않아도 결과 파일이 만들어집니다. 다만 Colab 에서는 맨 위의 자료 내려받기 셀을 먼저 실행해야 합니다. 스크립트와 데이터 파일을 그 셀이 내려받기 때문입니다.

**날마다 자동으로 실행하려면 — 예약 실행**
- 명령 한 줄로 실행되는 스크립트는, 컴퓨터가 정해진 시각에 대신 실행하도록 **예약**할 수 있습니다. 이를 **예약 실행(스케줄링, scheduling)** 이라고 합니다.
- 예약해 두면 아침마다 전날 기록까지 반영된 추천 파일이 만들어집니다.
- 예를 들면 「매일 새벽 2시에 `python recommend_batch.py` 를 실행하라」는 내용을 컴퓨터에 한 줄로 등록해 둡니다.
- **오늘 수업에서는 예약까지 하지 않고 개념만 소개합니다.** 오늘 실습은 명령 한 줄로 끝까지 실행되는 것까지입니다.

In [8]:
# 이 셀에서 하는 일 — 스크립트를 실행한다. 맨 앞의 ! 는 「이 줄은 컴퓨터에 내리는 명령」이라는 표시다
# 왜 하나 — 데이터 점검부터 저장까지 명령 한 줄로 끝나는지 확인한다

!python recommend_batch.py

① 데이터 점검 — 여섯 가지 모두 0건 · 통과


② 학습 — 상호작용 기록 8,086건 · 투자자 300명


③ 일괄 추천 — 3,000줄


④ 규격 검사 — 문제 0개 · 통과
⑤ 저장 — output/recommendations.csv · 300명 × 10개


**결과 읽는 법**

**결론: 셀 하나, 명령 한 줄로 ① \~ ⑤ 가 모두 끝났고 추천 3,000줄이 파일로 저장되었습니다.**
- 출력 다섯 줄이 단계 다섯 개입니다
  - **①** 데이터 점검 — 여섯 가지 모두 0건, 통과
  - **②** 학습 — 상호작용 **8,086건** 전체, 투자자 **300명**
  - **③** 일괄 추천 — 300명 × 10개 = **3,000줄**
  - **④** 규격 검사 — 문제 0개, 통과
  - **⑤** 저장 — `output/recommendations.csv`
- 서비스용은 300명입니다. 채점용 모델의 285명에 마지막 1개월에 처음 기록이 생긴 신규 15명이 더해집니다
- 6주차 노트북에서 같은 추천을 얻으려면 셀을 최소 7개 차례로 실행해야 했고, 파일 저장과 검사가 없었습니다

**추천 10개는 어떻게 계산되나 — `추천하기` 함수의 로직**

스크립트가 만든 3,000줄은 `추천하기` 함수를 투자자 300명에게 한 번씩 실행한 결과입니다. 한 명의 추천은 **다섯 단계**를 거칩니다.

1. **점수 계산** — 그 사람의 사용자 요인 × 아이템 요인 → 종목 100개 각각의 점수 (4주차 행렬분해)
2. **정렬** — 점수가 높은 순서로 종목을 늘어놓습니다
3. **위험도 상한** — 그 투자자의 투자 성향이 허용하는 위험도(안정형 3 · 성장형 4 · 공격형 5) 이하만 남깁니다
4. **이미 담은 종목 제외** — 이미 상호작용한 종목은 추천하지 않습니다
5. **위 10개** — 남은 종목의 위에서 10개를 점수와 함께 돌려줍니다 (4 와 5 는 `recsys.take` 가 합니다)

안정형 **U0001** 의 추천을 함수를 쓰지 않고 **단계별로 직접** 계산합니다. 마지막에 함수의 결과와 같은지 비교합니다.

In [9]:
# 이 셀에서 하는 일 — U0001 의 추천 10개를 다섯 단계로 직접 계산하고, 스크립트 함수의 결과와 비교한다
# 왜 하나 — 단계마다 종목이 몇 개 남는지 보면 위험도 상한과 이미 담은 종목 제외가 어떻게 작동하는지 알 수 있다

import numpy as np                                   # 숫자 계산 도구. 점수를 정렬할 때 쓴다

사람 = "U0001"
성향 = users.set_index("user_id")["persona"].to_dict()        # {투자자 번호: 투자 성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()   # {종목 번호: 위험도}
묶음 = recommend_batch.학습하기(interactions, items)            # 스크립트의 ② 학습과 같은 함수. 전체 기록으로 학습한다

점수 = 묶음["사용자_요인"][묶음["사용자_목록"].index(사람)] @ 묶음["아이템_요인"]   # 1) @ = 행렬 곱. 종목 100개의 점수
순서 = [묶음["아이템_목록"][자리] for 자리 in np.argsort(-점수)]                    # 2) 점수가 높은 순서의 종목 번호
통과 = [번호 for 번호 in 순서 if 위험도[번호] <= recommend_batch.상한[성향[사람]]]   # 3) 위험도 상한 이하만 남긴다
담은 = 묶음["이미_담은것"][사람]
남은 = [번호 for 번호 in 통과 if 번호 not in 담은]                                  # 4) 이미 담은 종목을 뺀다
직접 = [(번호, round(float(점수[묶음["아이템_목록"].index(번호)]), 4)) for 번호 in 남은[:10]]   # 5) 위에서 10개와 점수

print(f"{사람} · {성향[사람]} (위험도 상한 {recommend_batch.상한[성향[사람]]})")
print(f"1) 점수를 계산한 종목        : {len(점수)}개")
print(f"2) 점수가 가장 높은 3개      : {순서[:3]}")
print(f"3) 위험도 상한 이하          : {len(통과)}개")
print(f"4) 이미 담은 {len(담은)}개를 뺀 뒤    : {len(남은)}개")
print(f"5) 위에서 10개 중 앞 3개     : {직접[:3]}")
print("\n함수 결과와 같은가 :", 직접 == recommend_batch.추천하기(묶음, 사람, 성향, 위험도))

U0001 · 안정형 (위험도 상한 3)
1) 점수를 계산한 종목        : 100개
2) 점수가 가장 높은 3개      : ['I007', 'I006', 'I019']
3) 위험도 상한 이하          : 52개
4) 이미 담은 17개를 뺀 뒤    : 38개
5) 위에서 10개 중 앞 3개     : [('I002', 0.6211), ('I043', 0.4705), ('I014', 0.3974)]

함수 결과와 같은가 : True


**결과 읽는 법**

**결론: U0001 의 추천은 종목 100개가 「위험도 상한 52개 → 이미 담은 것 제외 38개 → 위 10개」로 줄어든 결과이고, 직접 계산한 값이 함수의 결과와 같습니다.**
- 점수가 가장 높은 3개(`I007` · `I006` · `I019`)는 이미 담은 종목이라 4) 에서 빠집니다
  - 그래서 1등 `I002` 는 **상한을 지키고 아직 담지 않은 종목 중 가장 높은 것**입니다
- 함수의 결과와 **같음**(`True`) — 함수 안에서 일어나는 일은 이 다섯 단계가 전부입니다
- 7.4 규격 검사의 「상한 초과 없음」 · 「이미 담은 종목 없음」은 이 3) · 4) 단계의 결과를 다시 검사하는 것입니다
- `학습하기` 가 만드는 사용자 요인 · 아이템 요인은 📖 7.8 에서 봅니다

**날마다 새로 만들어야 하는 이유 — 숫자로 확인하기**

새 상호작용이 추가되면 추천이 얼마나 바뀌는지 확인합니다.
- 상호작용 기록에서 **마지막 7일치를 뺀 데이터**로 만든 추천과, **전체 데이터**로 만든 추천을 비교합니다.
- 두 추천에서 10개 종목이 모두 같으면 바뀌지 않은 것이고, 하나라도 다르면 바뀐 것입니다.
- 아래 코드를 모두 이해할 필요는 없습니다. 셀을 실행하고, **결과 세 줄만 읽으면** 됩니다.

In [10]:
# 이 셀에서 하는 일 — 마지막 7일치 상호작용을 뺀 데이터로 만든 추천과, 전체 데이터로 만든 추천을 비교한다
# 왜 하나 — 새 상호작용이 며칠만 들어와도 추천이 얼마나 바뀌는지 숫자로 확인한다

성향 = users.set_index("user_id")["persona"].to_dict()             # 딕셔너리 {투자자 번호: 투자성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()        # 딕셔너리 {종목 번호: 1~5}

def 추천_목록(기록):
    """상호작용 기록으로 학습하고, 투자자 300명 각각의 추천 10개(종목 번호)를 딕셔너리로 돌려준다."""
    묶음 = recommend_batch.학습하기(기록, items)                    # 학습
    결과 = {}
    for 사람 in users["user_id"]:                                   # 300명을 한 명씩
        번호들 = []
        for 번호, 점수 in recommend_batch.추천하기(묶음, 사람, 성향, 위험도):
            번호들.append(번호)
        결과[사람] = 번호들
    return 결과

일주일_전 = interactions["ts"].max() - pd.Timedelta(days=7)         # 마지막 7일이 시작되기 직전 시각
예전_기록 = interactions[interactions["ts"] <= 일주일_전]            # 그때까지의 상호작용 기록
새_상호작용 = len(interactions) - len(예전_기록)                         # 마지막 7일에 새로 들어온 상호작용

예전_추천 = 추천_목록(예전_기록)                                    # 7일 전까지의 기록으로 만든 추천
지금_추천 = 추천_목록(interactions)                                 # 전체 기록으로 만든 추천

바뀐_사람 = 0                                                      # 추천이 한 개라도 바뀐 투자자 수
바뀐_종목 = 0                                                      # 바뀐 종목 수의 합
for 사람 in users["user_id"]:
    같은_개수 = len(set(예전_추천[사람]) & set(지금_추천[사람]))      # & = 두 목록에 모두 있는 종목
    if 같은_개수 < 10:
        바뀐_사람 = 바뀐_사람 + 1
    바뀐_종목 = 바뀐_종목 + (10 - 같은_개수)

print(f"마지막 7일에 새로 들어온 상호작용 : {새_상호작용}건")
print(f"추천이 한 개라도 바뀐 투자자 : {바뀐_사람}명 / {len(users)}명")
print(f"투자자 한 명당 바뀐 종목 : 평균 {바뀐_종목 / len(users):.2f}개 (10개 중)")

마지막 7일에 새로 들어온 상호작용 : 261건
추천이 한 개라도 바뀐 투자자 : 230명 / 300명
투자자 한 명당 바뀐 종목 : 평균 1.15개 (10개 중)


**결과 읽는 법**

**결론: 새 상호작용이 일주일치(261건)만 늘어도 300명 중 230명의 추천이 바뀝니다. 그래서 추천은 날마다 새로 만들어야 하고, 명령 한 줄로 실행되는 스크립트가 필요합니다.**
- 마지막 7일의 새 상호작용 **261건** — 전체 8,086건의 약 3%
- 추천이 한 개 이상 바뀐 투자자 **230명**(약 77%), 한 명당 평균 **1.15개**(10개 중)
- 우리 데이터는 하루 평균 약 22건뿐인데도 일주일이면 대부분의 추천이 달라집니다
- 사람이 매번 셀을 실행하기는 어렵습니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.4 결과 파일 확인 — 규격대로 만들어졌는가</mark>

**파이프라인에서의 위치:** ⑤ 저장 — 만들어진 파일이 ④ 에서 검사한 약속대로인지 눈으로 확인합니다.


**7.4 에서 하는 일** — 스크립트가 만든 파일을 **다시 읽어서** 규격대로 만들어졌는지 확인합니다. 안정형 투자자 U0001 의 추천 10개도 봅니다.

**왜 확인하나**
- 이 파일이 8주차 API 가 읽는 **유일한 입력**입니다. 열 이름이 하나만 달라도 API 가 오류를 냅니다.
- 그래서 만드는 쪽과 읽는 쪽이 **같은 규격**을 미리 정해 두고, 스크립트의 ④ 규격 검사가 지켰는지 자동으로 확인합니다.
- 금융에서는 이 파일에 **위험도 상한을 넘는 종목이 하나도 없어야** 합니다 (6주차 적합성원칙).

**규격 (이 수업에서 정한 약속)**

| 열 | 뜻 | 예 |
|---|---|---|
| `user_id` | 투자자 번호 (U0001 \~ U0300) | U0001 |
| `rank` | 추천 순위 (1 = 가장 먼저 추천, 10 = 열 번째). `score` 가 높은 순서 | 1 |
| `item_id` | 종목 번호 (I001 \~ I100) | I002 |
| `item_name` | 종목 이름 | TIGER 미국배당다우존스 |
| `risk_level` | 그 종목의 위험도 (1 = 가장 낮음, 5 = 가장 높음) | 2 |
| `persona` | 그 투자자의 투자 성향 (안정형 · 성장형 · 공격형). 추천 가능한 위험도 상한이 3 · 4 · 5 | 안정형 |
| `score` | 추천 점수. 행렬분해로 계산하며 높을수록 그 투자자에게 더 맞는다고 본 종목 (기록이 없는 투자자는 0) | 0.6211 |

- 열은 **이 이름, 이 순서**로 7개입니다. 투자자마다 **정확히 10줄**입니다.

**④ 규격 검사가 확인하는 다섯 가지** — `규격_검사` 함수의 로직

어긴 것이 있으면 **문제 목록에 문장이 하나씩 쌓이고**, 목록이 비어 있으면 통과입니다.

| 검사 | 코드가 하는 일 |
|---|---|
| 1. 열 이름과 순서가 규격과 같은가 | `list(표.columns) != 규격_열` |
| 2. 300명 모두 추천이 있는가 | 서로 다른 `user_id` 의 수(`nunique`)가 투자자 수와 같은지 |
| 3. 사람마다 정확히 10줄인가 | `groupby("user_id").size()` 가 모두 10 인지 |
| 4. 투자 성향의 상한을 넘는 종목이 없는가 (안정형 3 · 성장형 4 · 공격형 5) | 줄마다 `risk_level` 이 그 투자 성향의 상한 이하인지 |
| 5. 이미 담은 종목을 추천하지 않았는가 | 줄마다 `item_id` 가 그 투자자가 이미 담은 종목에 없는지 |

- 이 검사가 실제로 문제를 찾아내는지는 아래 7.4 의 마지막 셀에서 확인합니다.

**파일 형식**
- **CSV(comma-separated values)** — 값을 쉼표로 나누어 적은 글자 파일입니다. 엑셀 · 메모장 · 파이썬에서 모두 열 수 있습니다.
- **인코딩(encoding)** — 글자를 컴퓨터의 숫자로 바꾸는 방식입니다. `utf-8-sig` 로 저장하면 엑셀에서 열어도 한글이 깨지지 않습니다.
- **경로(path)** — 파일의 위치입니다. `output/recommendations.csv` 는 현재 폴더 안의 `output` 폴더에 있는 `recommendations.csv` 라는 뜻입니다.
- **엑셀에서 열어 보기** — Colab 왼쪽의 파일 아이콘(📁)에서 `notebooks/output/recommendations.csv` 를 내려받아 엑셀로 열 수 있습니다. `persona` 열로 거르면 안정형 투자자의 추천만 볼 수 있습니다.

In [11]:
# 이 셀에서 하는 일 — 스크립트가 만든 결과 파일을 읽어 크기 · 열 · U0001 의 추천을 본다
# 왜 하나 — 다음 주 API 가 읽을 파일이 규격대로 나왔는지 눈으로 확인한다

import os                                                                 # 파일이 있는지 확인하는 도구
assert os.path.exists("output/recommendations.csv"), "결과 파일이 아직 없습니다. 위의 7.3 셀(!python recommend_batch.py)을 먼저 실행하세요"   # 없으면 이 메시지를 보여 주고 실행을 중단한다

결과 = pd.read_csv("output/recommendations.csv", encoding="utf-8-sig")   # 저장된 파일을 다시 읽는다

print(f"크기 {결과.shape}  ← 3,000줄 × 열 7개")
print("열 :", list(결과.columns))
print(f"안정형 투자자에게 간 추천 중 가장 높은 위험도 : {결과[결과['persona'] == '안정형']['risk_level'].max()}")
결과[결과["user_id"] == "U0001"]                     # U0001 의 10줄만

크기 (3000, 7)  ← 3,000줄 × 열 7개
열 : ['user_id', 'rank', 'item_id', 'item_name', 'risk_level', 'persona', 'score']
안정형 투자자에게 간 추천 중 가장 높은 위험도 : 3


,user_id,rank,item_id,item_name,risk_level,persona,score
0,U0001,1,I002,TIGER 미국배당다우존스,2,안정형,0.6211
1,U0001,2,I043,Vanguard FTSE Developed Markets ETF,3,안정형,0.4705
2,U0001,3,I014,KT,2,안정형,0.3974
3,U0001,4,I011,KODEX 단기채권,1,안정형,0.3722
4,U0001,5,I021,Verizon Communications,2,안정형,0.3690
5,U0001,6,I015,LG유플러스,2,안정형,0.3645
6,U0001,7,I033,TIGER 은행고배당플러스TOP10,2,안정형,0.3356
7,U0001,8,I020,Johnson & Johnson,2,안정형,0.3127
8,U0001,9,I035,TIGER 200,3,안정형,0.3036
9,U0001,10,I055,Apple,3,안정형,0.2801


**결과 읽는 법**

**결론: 저장된 파일은 규격과 같고(3,000줄 × 7열), 안정형에게 간 추천의 위험도는 모두 3 이하입니다.**
- 크기 `(3000, 7)` — 300명 × 10개 = 3,000줄, 열 7개. 열 이름과 순서가 규격과 같습니다
- 안정형 투자자 전체의 추천 중 가장 높은 위험도가 **3** → 안정형 상한(3)을 넘는 추천이 없습니다
- 아래 표는 안정형 **U0001** 의 추천 10개입니다
  - `rank` 가 1 부터 10 까지이고 `risk_level` 은 모두 3 이하입니다
  - 6주차 6.2 에서 4주차 모델이 추천했던 위험도 4 \~ 5 종목(TIGER 2차전지테마 · 크래프톤 등)이 없습니다

**규격 검사가 위험도 상한 위반을 찾아내는지 확인하기**

④ 규격 검사는 지금까지 통과만 했습니다. 이번에는 결과 표를 복사해서 **안정형 U0001 의 1등 종목의 위험도를 일부러 5 로 바꾼 뒤** 규격 검사에 넘깁니다.
- 안정형의 위험도 상한은 3 입니다. 5 는 상한을 넘으므로 6주차의 적합성원칙을 어긴 표가 됩니다.
- 저장된 파일과 원래 표는 바꾸지 않습니다.
- 이 셀의 코드도 모두 이해할 필요는 없습니다. **결과 두 줄만 읽으면** 됩니다.

In [12]:
# 이 셀에서 하는 일 — 결과 표를 복사해 U0001 의 1등 위험도를 5 로 바꾸고, 규격 검사에 넘긴다
# 왜 하나 — 규격 검사가 안정형에게 위험도 상한(3)을 넘는 추천을 찾아내는지 직접 확인한다. 저장된 파일은 바꾸지 않는다

학습_결과 = recommend_batch.학습하기(interactions, items)            # 스크립트와 같은 학습
정상_표 = recommend_batch.일괄_추천(학습_결과, users, items)          # 스크립트와 같은 300명 × 10개 표
print("정상인 표 :", recommend_batch.규격_검사(정상_표, users, 학습_결과) or "문제 없음")

잘못된_표 = 정상_표.copy()                                            # 복사본만 바꾼다
잘못된_표.loc[잘못된_표.index[0], "risk_level"] = 5                   # 맨 첫 줄(U0001 의 1등)의 위험도를 5 로
print("바꾼 표 :", recommend_batch.규격_검사(잘못된_표, users, 학습_결과))

정상인 표 : 문제 없음
바꾼 표 : ['U0001 에게 상한을 넘는 I002']


**결과 읽는 법**

**결론: 규격 검사는 정상 표를 통과시키고, 위험도를 일부러 5 로 바꾼 표에서는 그 줄을 찾아냅니다. 상한을 넘는 추천은 파일로 저장되지 않습니다.**
- **정상인 표** — 문제를 찾지 못함 (스크립트의 ④ 통과와 같은 결과)
- **바꾼 표** — `U0001 에게 상한을 넘는 I002` 라고 알려 줍니다
  - I002 의 원래 위험도는 2 인데 5 로 바꾸니 안정형 상한 3 을 넘었습니다
- 스크립트는 ④ 에서 이런 문제가 나오면 ⑤ **저장하기 전에** 중단합니다

**🗣 수업에서 함께 보기**

6주차까지는 셀을 7개 이상 차례로 실행해야 얻던 추천을(그것도 화면에만 나왔습니다), 오늘은 **명령 한 줄**로 파일까지 만들었습니다. 그런데 추천 방법과 점수(**0.3899**)는 6주차와 같습니다. 점수가 같다는 것은 7.5 에서 직접 확인할 수 있습니다.

점수가 그대로라면, 오늘 무엇이 나아진 걸까요? 금융 회사의 입장에서 생각해 보세요.

- 힌트 — 7.1 의 데이터 점검, 7.3 에서 본 「일주일치 상호작용만 늘어도 추천이 바뀌는 숫자」, 7.4 에서 본 규격 검사, 사람이 셀을 직접 실행할 때 생길 수 있는 실수를 떠올려 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.5 같은 방법인지 채점 — 6주차 0.3899 와 비교한다</mark>

> 📖 **혼자 읽기** — 수업에서는 PPT 로 설명하고, 이 절은 복습할 때 직접 실행해 보는 부분입니다.
**파이프라인에서의 위치:** 파이프라인 밖 확인 — 5단계로 묶은 추천이 6주차와 같은 점수(0.3899)가 나오는지 봅니다.


**7.5 에서 하는 일** — 스크립트 안의 함수로 **채점용**(학습 구간만) 추천을 만들고 Recall@10 을 계산합니다. 6주차와 같은 **0.3899** 가 나와야 합니다.

**왜 하나**
- 셀을 파일로 옮기는 과정에서 무언가 빠지거나 바뀌지 않았는지 확인해야 합니다.
- 같은 기준으로 채점해 같은 점수가 나오면, 옮긴 결과가 정확하다는 증거입니다.
- 코드를 옮기거나 정리하는 일(**리팩터링, refactoring**)은 실무에서 흔합니다. 정리한 뒤에는 **결과가 그대로인지 숫자로 확인**합니다.

**순서**
1. 스크립트의 `학습하기` 로 **학습 구간만** 사용해 학습한다
2. 스크립트의 `추천하기` 로 한 사람에게 추천할 (종목 번호, 점수) 10쌍을 구한다
3. 채점 함수가 요구하는 모양(종목 번호 목록)으로 바꾼다
4. 3주차부터 쓰던 `recsys.recall_at_k` 로 채점한다

- `recommend_batch.학습하기` 처럼 「파일 이름.함수 이름」으로 쓰면, 스크립트에 정의된 함수를 다시 작성하지 않고 노트북에서 **호출**(실행)할 수 있습니다.
- 점수는 **채점용**으로만 잴 수 있습니다. 서비스용 추천은 맞혀 볼 미래가 없습니다.

In [13]:
# 이 셀에서 하는 일 — 스크립트의 함수로 채점용 추천을 만들어 Recall@10 을 잰다
# 왜 하나 — 셀을 파일 하나로 옮겨도 6주차와 같은 추천인지 0.3899 로 확인한다

묶음_채점용 = recommend_batch.학습하기(train, items)                # 학습 구간만으로 학습
성향 = users.set_index("user_id")["persona"].to_dict()             # 딕셔너리 {투자자 번호: 투자성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()        # 딕셔너리 {종목 번호: 1~5}

def 채점용_추천(사람, 이미):                                        # 채점 함수가 요구하는 모양 (사람, 이미)
    """스크립트의 추천하기가 준 (번호, 점수) 에서 번호만 꺼낸다."""
    번호들 = []
    for 번호, 점수 in recommend_batch.추천하기(묶음_채점용, 사람, 성향, 위험도):
        번호들.append(번호)
    return 번호들

채점_점수, 인원 = recsys.recall_at_k(채점용_추천, train, test)      # 3주차부터 쓰던 채점 함수
print(f"스크립트의 함수로 만든 추천 — Recall@10 = {채점_점수:.4f}   (채점 대상 {인원}명 · 6주차 0.3899)")

스크립트의 함수로 만든 추천 — Recall@10 = 0.3899   (채점 대상 266명 · 6주차 0.3899)


**결과 읽는 법**

**결론: 스크립트의 함수로 만든 추천도 Recall@10 = 0.3899 로 6주차와 같습니다. 코드를 파일로 옮기면서 바뀐 것이 없다는 확인입니다.**
- 6주차 위험도 상한 추천과 소수점 넷째 자리까지 같고, 채점 대상도 같은 **266명**입니다
- 같은 방법으로 만든 채점용 추천은 6주차와 하나도 바뀌지 않았습니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.6 학습과 서빙 — 학습은 오래 걸리고 추천은 빠르다</mark>

> 📖 **혼자 읽기** — 수업에서는 PPT 로 설명하고, 이 절은 복습할 때 직접 실행해 보는 부분입니다.

**7.6 에서 하는 일** — 학습 한 번과 추천 한 번이 각각 얼마나 걸리는지 **시간을 재서** 비교합니다.

**왜 나누나**
- 학습은 오래 걸려서 가끔 실행합니다. 학습해 둔 결과로 한 사람의 추천만 계산하는 일은 빠르고 자주 실행합니다.
- 고객이 앱을 열 때마다 학습을 다시 하면 기다리는 시간이 길어집니다. 그래서 모델 학습은 미리 해 두고, 고객 요청에는 미리 만든 학습 결과로 추천을 제공합니다.
- 8주차의 API 는 한 걸음 더 나아가, 7.3 에서 **미리 만든 결과 파일을 읽어서** 응답합니다. 학습을 하지 않는다는 점은 같습니다.

**주의할 점**
- 시간은 컴퓨터마다, 실행할 때마다 조금씩 다릅니다. **정확한 값보다 몇 배 차이인지**를 봅니다.
- `time.perf_counter()` 는 현재 시각을 아주 정밀하게 재는 함수입니다. 끝난 시각에서 시작 시각을 빼면 걸린 시간입니다.

In [14]:
# 이 셀에서 하는 일 — 학습 한 번과, 300명에게 추천을 제공하는 데 걸린 시간을 잰다
# 왜 하나 — 학습과 서빙을 왜 나누는지 숫자로 확인한다

import time                                               # 시간을 재는 도구

시작 = time.perf_counter()                                # 시작 시각
recommend_batch.학습하기(train, items)                     # 학습 한 번
학습_시간 = time.perf_counter() - 시작                     # 끝난 시각 - 시작 시각

시작 = time.perf_counter()
for 사람 in users["user_id"]:                             # 투자자 300명에게 각각 한 번씩
    recommend_batch.추천하기(묶음_채점용, 사람, 성향, 위험도)   # 서빙 — 모델을 다시 학습하지 않고, 이미 만든 학습 결과로 해당 투자자의 추천만 계산한다
서빙_시간 = (time.perf_counter() - 시작) / len(users)       # 한 사람당 평균

print(f"학습 한 번 {학습_시간 * 1000:.1f}ms · 한 사람 추천 {서빙_시간 * 1000:.3f}ms · 약 {학습_시간 / 서빙_시간:,.0f}배")

학습 한 번 46.5ms · 한 사람 추천 0.035ms · 약 1,317배


**결과 읽는 법**

**결론: 학습 한 번이 추천 한 번보다 수백 배 이상 오래 걸립니다. 그래서 학습은 가끔 하고, 학습해 둔 결과로 하는 추천은 자주 합니다.**
- 첫 숫자는 **학습 한 번**, 둘째는 **한 사람의 추천 한 번**, 마지막은 **몇 배**입니다 (ms = 1,000분의 1초)
- 실행할 때마다 값은 조금 다르지만 「수백 배 이상」은 같습니다
- 우리 데이터는 종목 100개 · 투자자 300명뿐이라 둘 다 빠릅니다. 데이터가 커지면 학습 시간이 훨씬 길어집니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.7 재현성 — 두 번 실행해도 같은 파일이 나온다</mark>

> 📖 **혼자 읽기** — 복습할 때 직접 실행해 보는 부분입니다.

**7.7 에서 하는 일** — 스크립트를 **한 번 더** 실행하고, 두 결과 파일의 내용이 바이트 단위까지 같은지 비교합니다.

**왜 확인하나**
- 날마다 자동으로 실행하는 스크립트가 같은 기록으로 **다른 결과**를 내면, 어느 쪽이 맞는지 알 수 없습니다.
- 「어제와 추천이 달라졌다」는 문의가 왔을 때, 데이터가 바뀐 것인지 스크립트의 결과가 불안정한 것인지 가려낼 수 있어야 합니다.
- 행렬분해에는 무작위 요소가 있습니다. `random_state=42` 로 고정했기 때문에 같은 입력에서 같은 결과를 얻습니다. 6주차 FM 의 `seed=42` 와 같은 역할을 합니다.

**아래 셀에서 쓰는 도구**
- `shutil.copy` — 파일을 복사합니다.
- `filecmp.cmp(..., shallow=False)` — 두 파일을 **내용까지** 비교합니다. 같으면 True.

In [15]:
# 이 셀에서 하는 일 — 스크립트를 한 번 더 실행하고, 두 결과 파일이 똑같은지 비교한다
# 왜 하나 — 자동으로 매일 돌릴 스크립트는 같은 입력에 같은 결과를 내야 믿을 수 있다

import shutil, filecmp                                     # 파일 복사 도구 · 파일 비교 도구

import os
assert os.path.exists("output/recommendations.csv"), "결과 파일이 아직 없습니다. 7.3 셀(!python recommend_batch.py)을 먼저 실행하세요"   # 없으면 이 메시지를 보여 주고 실행을 중단한다

shutil.copy("output/recommendations.csv", "output/first_run.csv")       # 첫 결과를 옆에 복사해 둔다
!python recommend_batch.py > /dev/null
같은_파일 = filecmp.cmp("output/recommendations.csv", "output/first_run.csv", shallow=False)   # 내용까지 비교
print(f"두 번 실행한 결과 파일이 바이트 하나까지 같은가 : {같은_파일}")

두 번 실행한 결과 파일이 바이트 하나까지 같은가 : True


**결과 읽는 법**

**결론: 스크립트를 두 번 실행한 결과 파일이 바이트 하나까지 같아서(`True`) 재현성이 확인되었습니다.**
- **재현성** — 같은 상호작용 기록으로 실행하면 같은 결과 파일이 나오는 성질
- 둘째 줄의 `> /dev/null` 은 스크립트의 출력 다섯 줄을 화면에 표시하지 않고 버린다는 뜻입니다 (출력 내용은 7.3 에서 확인함)

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.8 함수 안에서 하는 일 — 학습하기와 나머지</mark>

**파이프라인에서의 위치:** ① \~ ④ 의 **함수 안** — 7.2 에서 본 `main` 이 부르는 함수들이 실제로 어떻게 계산하는지 풉니다.

**7.8 에서 하는 일** — `main` 이 부르는 함수 중 **`학습하기` · `일괄_추천`** 과 기록이 없는 투자자의 추천을 직접 실행하면서 중간 결과를 확인합니다.

**나머지 함수의 로직은 해당 단계의 절에 있습니다**
- `데이터_점검` — 7.1
- `추천하기` (점수 → 정렬 → 위험도 상한 → 이미 담은 종목 제외 → 위 10개) — 7.3
- `규격_검사` — 7.4

**함수가 들어 있는 파일**

| 파일 | 함수 | 하는 일 | 이 노트북에서 |
|---|---|---|---|
| `recommend_batch.py` | `데이터_점검` | ① 문제가 있는 줄을 찾아 개수를 센다 | 7.1 |
| `recommend_batch.py` | `학습하기` | ② 행렬분해 모델을 만든다 | **7.8** |
| `recommend_batch.py` | `추천하기` | 한 투자자의 추천 10개를 계산한다 | 7.3 · 7.5 · 7.8 |
| `recommend_batch.py` | `일괄_추천` | ③ 300명 모두에게 `추천하기` 를 실행해 표를 만든다 | 7.3 · **7.8** |
| `recommend_batch.py` | `규격_검사` | ④ 결과 표가 규격을 지켰는지 검사한다 | 7.4 |
| `recsys.py` | `interaction_matrix` | 사용자 × 종목 행렬을 만든다 | **7.8** |
| `recsys.py` | `take` | 이미 담은 종목을 빼고 위에서 10개를 고른다 | 7.3 |
| `recsys.py` | `load` · `split_by_time` · `recall_at_k` | 데이터 읽기 · 시간순 분할 · 채점 | 준비 · 7.5 |

- 두 파일 모두 노트북 셀이 아니라 **파일**입니다. 노트북에서는 `import` 로 불러 쓰고, 코드를 글자 그대로 보려면 `inspect.getsource(함수)` 를 사용합니다.


#### ② 학습하기 — 행렬분해 모델을 만든다

**순서**
1. `recsys.interaction_matrix` 가 **사용자 × 종목 행렬**을 만듭니다. 상호작용이 있으면 1, 없으면 0 입니다.
2. `TruncatedSVD(n_components=8)` 로 **행렬분해**를 합니다. 행렬 하나가 두 개로 나뉩니다.
   - **사용자 요인** — 투자자마다 요인 8개 (300 × 8)
   - **아이템 요인** — 종목마다 요인 8개 (8 × 100)
3. 추천할 때 필요한 것을 함께 저장합니다 — 인기순위 · 투자자별로 이미 담은 종목 · 번호 목록.

`random_state=42` 로 무작위 요소를 고정하므로, 다시 실행해도 같은 모델이 나옵니다(7.7).

In [16]:
# 이 셀에서 하는 일 — 학습하기가 만드는 행렬과 저장하는 것을 하나씩 확인한다
# 왜 하나 — 함수 안에서 어떤 값이 만들어지는지 눈으로 본다

행렬 = recsys.interaction_matrix(interactions, items)        # 1단계 — 사용자 × 종목 행렬
print(f"사용자 × 종목 행렬 : {행렬.shape} · 값이 1 인 곳 {int(행렬.values.sum()):,}개")

묶음 = recommend_batch.학습하기(interactions, items)          # 2 ~ 3단계 — 행렬분해와 저장
print("\n학습하기가 돌려주는 것")
for 이름, 값 in 묶음.items():
    크기 = 값.shape if hasattr(값, "shape") else f"{len(값)}개"
    print(f"  {이름:8s} {크기}")
print(f"\n인기순위 앞 3개 : {묶음['인기순위'][:3]}")
print(f"U0001 이 이미 담은 종목 : {len(묶음['이미_담은것']['U0001'])}개")

사용자 × 종목 행렬 : (300, 100) · 값이 1 인 곳 8,086개



학습하기가 돌려주는 것
  사용자_요인   (300, 8)
  아이템_요인   (8, 100)
  사용자_목록   300개
  아이템_목록   100개
  인기순위     100개
  이미_담은것   300개

인기순위 앞 3개 : ['I042', 'I043', 'I040']
U0001 이 이미 담은 종목 : 17개


**결과 읽는 법**

**결론: `학습하기` 는 사용자 × 종목 행렬(300 × 100)을 사용자 요인(300 × 8)과 아이템 요인(8 × 100)으로 나누고, 추천에 필요한 목록을 함께 저장합니다.**
- 행렬에서 1 인 곳 **8,086개** = 상호작용 기록 8,086건
- 사용자 요인 × 아이템 요인 = 300 × 100 **점수표** (4주차의 「사용자 요인 × 아이템 요인 = 추천 점수」)
- 사용자 목록 · 아이템 목록 — 요인 행렬의 몇 번째 줄이 누구인지 알려 주는 번호 목록
- 인기순위 — 상호작용이 많은 종목 순서. 기록이 없는 투자자에게 씁니다 (아래)

#### ③ 추천하기 (이어서) — 기록이 없는 투자자는?

추천하기의 다섯 단계는 7.3 에서 봤습니다. 기록이 없는 투자자는 사용자 요인이 없으므로 점수를 계산할 수 없습니다. 이때는 `인기순위` 를 대신 씁니다(**콜드 스타트, cold start**). 위험도 상한과 이미 담은 종목 제외는 똑같이 적용하고, 점수는 0 으로 적습니다.

오늘의 서비스용 모델은 300명 모두에게 기록이 있어서 이 경우가 없습니다. 채점용 모델(학습 구간만 사용)에는 있어서, 그 모델로 확인합니다.

In [17]:
# 이 셀에서 하는 일 — 학습 구간에 기록이 없는 투자자에게 추천이 어떻게 나오는지 본다
# 왜 하나 — 기록이 없는 사람에게는 점수 대신 인기순위를 쓴다는 것을 확인한다

성향 = users.set_index("user_id")["persona"].to_dict()           # {투자자 번호: 투자 성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()      # {종목 번호: 위험도}

채점용_묶음 = recommend_batch.학습하기(train, items)          # 채점용 — 학습 구간(7,095건)만 사용
기록_없음 = [사람 for 사람 in users["user_id"] if 사람 not in 채점용_묶음["사용자_목록"]]
print(f"학습 구간에 기록이 없는 투자자 : {len(기록_없음)}명 · 예) {기록_없음[:3]}")

사람 = 기록_없음[0]
결과 = recommend_batch.추천하기(채점용_묶음, 사람, 성향, 위험도)
print(f"\n{사람} · {성향[사람]} 의 추천 {len(결과)}개 중 앞 3개 (종목 번호, 점수) : {결과[:3]}")
print(f"인기순위 앞 3개 : {채점용_묶음['인기순위'][:3]}")

학습 구간에 기록이 없는 투자자 : 15명 · 예) ['U0009', 'U0015', 'U0026']

U0009 · 공격형 의 추천 10개 중 앞 3개 (종목 번호, 점수) : [('I042', 0.0), ('I043', 0.0), ('I040', 0.0)]
인기순위 앞 3개 : ['I042', 'I043', 'I040']


**결과 읽는 법**

**결론: 학습 구간에 기록이 없는 투자자 15명(3주차의 신규 투자자)에게는 점수 대신 인기순위로 추천하고, 점수는 0.0 입니다.**
- 점수를 계산하지 않았기 때문에 점수가 모두 **0.0** 입니다
- 종목 순서는 인기순위와 같습니다. U0009 는 공격형(상한 5)이라 걸러진 종목이 없고, 안정형이라면 상한 3 을 넘는 종목이 빠집니다

#### ③ 일괄_추천 — 300명 모두에게 추천하기를 실행해 표를 만든다

**순서**
1. 투자자 300명을 한 명씩 돌면서 `추천하기` 를 실행합니다.
2. 나온 (종목 번호, 점수) 10쌍에 `rank` 를 1 부터 붙이고, 종목 이름 · 위험도 · 투자 성향을 더해 **한 줄씩** 만듭니다.
3. 모든 줄을 모아 **규격의 열 이름 · 순서**로 표 하나를 만듭니다. 300명 × 10줄 = 3,000줄입니다.

In [18]:
# 이 셀에서 하는 일 — 일괄_추천 으로 결과 표를 만들고, 크기와 투자자별 줄 수를 확인한다
# 왜 하나 — 300명 × 10줄 = 3,000줄이 규격대로 만들어지는지 본다

표 = recommend_batch.일괄_추천(묶음, users, items)
print("표의 크기 (줄, 열)     :", 표.shape)
print("투자자별 줄 수          :", 표.groupby("user_id").size().unique())   # unique = 서로 다른 값. 10 하나만 나와야 한다

표의 크기 (줄, 열)     : (3000, 7)
투자자별 줄 수          : [10]


**결과 읽는 법**

**결론: `일괄_추천` 이 만든 표는 3,000줄 × 7열이고, 300명 모두 정확히 10줄입니다.**
- 7.3 에서 스크립트가 저장한 파일과 같은 규격입니다
- 투자자별 줄 수의 값이 **10 하나뿐**입니다

**정리 — 함수마다 핵심 로직 한 줄**

| 함수 | 핵심 로직 | 어디서 |
|---|---|---|
| `데이터_점검` | 문제가 있는 줄을 `True` 로 표시하고 `sum()` 으로 개수를 센다 | 7.1 |
| `학습하기` | 사용자 × 종목 행렬을 행렬분해해서 사용자 요인 · 아이템 요인을 만든다 | 7.8 |
| `추천하기` | 사용자 요인 × 아이템 요인 → 점수 → 정렬 → 위험도 상한 → 이미 담은 종목 제외 → 위 10개 | 7.3 |
| `일괄_추천` | 300명에게 `추천하기` 를 실행해 규격의 표를 만든다 | 7.8 |
| `규격_검사` | 결과 표가 열 · 인원 · 줄 수 · 위험도 상한 · 이미 담은 종목 규격을 지켰는지 검사한다 | 7.4 |

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.9 오늘의 마무리</mark>

1. 6주차 최종 추천(행렬분해 + 위험도 상한)을 **5단계 파이프라인**으로 만들었습니다 — 데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장.
2. **데이터 점검** 여섯 가지가 모두 0건이었고, 문제가 있는 데이터를 일부러 넣어 보니 점검이 찾아냈습니다.
3. 이 모든 일이 `recommend_batch.py` **명령 한 줄**로 끝났고, 결과 파일 3,000줄이 규격 검사를 통과했습니다.
4. 같은 함수로 채점하면 **0.3899** 로 6주차와 같습니다. 두 번 실행해도 같은 파일이 나왔습니다(**재현성**).

**오늘 기억할 한 줄** — **추천 방법과 점수는 그대로이고, 추천을 만드는 일을 사람이 직접 하지 않아도 됩니다.**

다음 주에는 이 결과 파일을 읽어 **추천을 응답하는 API(FastAPI)** 를 만듭니다. 투자자 번호를 보내면 그 투자자의 추천 10개를 응답합니다.

In [19]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 점수가 그대로인 주차도 적는다. 무엇이 바뀌었는지 말할 수 있으면 된다

recsys.record(7, "추천 파이프라인 · 실행 한 번", 채점_점수,
              note="데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장을 스크립트 하나로. 점수는 그대로, 사람이 직접 실행하지 않아도 된다")

recsys.leaderboard(upto=7)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 7 · 추천 파이프라인 · 실행 한 번 · Recall@10 = 0.3899


,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 생성 + 순위 매기기,0.3382,2주차 세그먼트로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다
5,6,투자 성향별 위험도 상한,0.3899,4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 205개 → 0개
6,7,추천 파이프라인 · 실행 한 번,0.3899,데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장을 스크립트 하나로. 점수...


**결과 읽는 법**

**결론: 점수판에 7주차 Recall@10 = 0.3899 가 기록되었고, 6주차와 같습니다.**
- 아래 표가 점수판입니다. 새로 연 환경에서는 **1주차와 7주차 두 줄만** 보입니다
  - 점수판은 그 주차의 노트북을 실행한 환경에 쌓입니다. 같은 Colab 환경에서 다른 주차 노트북도 실행했다면 그 줄도 보입니다
- `note` 끝의 `...` 은 글이 길어서 화면에서 잘린 것입니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 7.1 결과**

어느 날 데이터 점검에서 **「똑같은 상호작용 기록 10건」** 이 나왔다고 가정해 봅시다. 스크립트는 지금처럼 **실행을 중단해야** 할까요, 아니면 10건만 지우고 **계속 진행해야** 할까요?

- 힌트 — 중단하면 고객은 어제 만든 추천을 그대로 보고, 진행하면 새 추천을 봅니다. 중복이 왜 생겼는지 모르는 상태에서 어느 쪽이 더 위험할까요?
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 7.3 결과**

채점할 때는 **학습 구간만**으로 학습했는데, 스크립트는 **상호작용 기록 전체**로 학습했습니다.

3주차에서 「미래 기록으로 배우면 데이터 누수」라고 배웠는데, 이건 괜찮을까요? 그리고 두 모델의 추천은 얼마나 다를까요?

- 힌트 — 서비스용 추천은 **언제** 고객에게 보여 주는 것인지 생각해 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 3 · 7.4 결과**

④ 규격 검사를 **하지 않고** 파일을 다음 주 API 에 넘긴다고 해 봅시다.

다섯 가지 검사 가운데 하나라도 어긴 파일이 넘어가면, 고객이 받는 추천에는 각각 어떤 일이 생길까요?

- 힌트 — 7.4 의 「왜 확인하나」와 6주차의 적합성원칙을 떠올려 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북과 같은 코드로 실제로 계산한 값입니다.

**함께 보기 답 — 추천은 같지만, 추천을 만드는 일이 실수 없이 · 예약해서 실행할 수 있게 · 검사를 거쳐 실행되게 되었습니다**

- **셀을 직접 실행하는 과정에서 생길 수 있는 실수를 줄입니다**
  - 셀을 직접 실행하다 보면 순서를 틀리거나 한 셀을 빠뜨릴 수 있습니다. 스크립트는 언제나 같은 순서로 실행됩니다.
- **예약 실행으로 정해진 시각에 실행할 수 있습니다**
  - 7.3 에서 본 것처럼, 상호작용이 일주일치(261건)만 늘어도 300명 중 230명의 추천이 바뀝니다. 그래서 추천은 날마다 새로 만들어야 합니다.
  - 명령 한 줄이라, 정해진 시각에 실행되도록 예약할 수 있습니다(예약 실행, scheduling). 예약해 두면 아침마다 전날 기록까지 반영된 추천 파일을 만들 수 있습니다. 오늘 수업에서는 예약까지 하지 않고 개념만 소개합니다.
- **잘못된 추천이 고객에게 전달되지 않게 합니다**
  - 들어오는 데이터(데이터 점검)와 나가는 파일(규격 검사)을 코드가 검사하고, 문제가 있으면 실행을 중단합니다.
  - 7.4 에서 본 것처럼, 규격 검사는 안정형 U0001 의 추천에 위험도 5 가 섞인 표를 찾아냈습니다. 이런 파일은 저장되지 않습니다 — 적합성원칙(6주차).
- **다음 단계에서 그대로 사용할 수 있습니다**
  - 규격에 맞는 파일이 만들어지므로, 다음 주 API 가 이 파일을 읽어서 사용할 수 있습니다.
- **한 줄로 정리하면** — 추천 성능(점수)은 그대로이고, 추천을 만드는 과정이 **실수 없이 · 검사를 거쳐 · 파일로 저장되도록** 바뀌었습니다. 실제 서비스에서 추천 모델을 사용하려면 추천 성능과 이런 운영 방식이 모두 갖춰져야 합니다.

**혼자 해 보기 1 답 — 실행을 중단하는 쪽이 안전합니다. 원인을 모르는 채로 일부만 고치면 다른 문제를 발견하지 못할 수 있기 때문입니다**

- 중복 10건은 결과일 뿐, **왜 생겼는지**는 모릅니다.
  - 전송이 두 번 된 것일 수도 있고, 기록을 만드는 시스템 자체에 오류가 생긴 것일 수도 있습니다.
  - 후자라면 중복 말고도 오류가 있는 기록이 더 있을 수 있습니다. 10건만 지우고 진행하면 나머지 오류를 발견하지 못합니다.
- 중단해도 **큰 손해가 없습니다.** 고객은 어제 만든 추천을 하루 더 받습니다.
- 반대로 오류가 있는 데이터로 진행하면 **오류 메시지 없이 틀린 결과**로 추천 3,000개가 만들어질 수 있습니다. 금융에서는 성향에 맞지 않는 추천으로 이어질 수도 있습니다.
- 실무에서는 실행을 중단하고 담당자에게 알린 뒤, 원인을 확인하고 다시 실행하는 것이 기본입니다. 원인이 확인된 경우에만 해당 데이터를 자동으로 삭제하고 진행하도록 별도 규칙을 둡니다.
- **한 줄로 정리하면** — 원인을 모르면 실행을 중단한다. 하루 늦은 추천이 틀린 추천보다 낫습니다.

**혼자 해 보기 2 답 — 괜찮습니다. 서비스용 추천을 보여 주는 시점에는 그 기록이 모두 과거이기 때문입니다**

- 데이터 누수는 **채점할 때** 생기는 문제입니다.
  - 채점은 「모르는 미래를 얼마나 맞히나」를 재는 일이라, 맞힐 기간을 학습에 넣으면 점수가 실제 실력보다 높게 나옵니다(3주차).
- 서비스용 추천은 **내일 고객에게 보여 줄** 추천입니다.
  - 그 시점에는 마지막 한 달도 이미 지난 과거입니다. 가장 최근 기록까지 모두 쓰는 것이 맞습니다.
  - 그래서 실무에서는 **채점용 모델로 실력을 재고, 서비스용 모델은 전체 기간으로 다시 학습**합니다.
- 두 모델의 추천은 꽤 비슷하지만 같지는 않습니다.
  - 300명의 추천 10개를 비교하면 평균 **7.45개**가 같고, 10개가 모두 같은 사람은 **11명**입니다.
  - 채점용에 없던 **신규 15명**도 서비스용에서는 자기 기록으로 추천을 받습니다.
- **한 줄로 정리하면** — 추천 성능은 미래 기록을 학습에서 제외하고 평가하고, 서비스용 모델은 그 시점까지 확보한 기록 전체로 학습합니다.

**혼자 해 보기 3 답 — API 에 오류가 나거나, 추천이 비거나, 고객에게 맞지 않는 종목이 전달됩니다**

- **열 이름 · 순서가 다르면** — API 가 `item_name` 같은 열을 찾지 못해 오류가 납니다. 그 API 를 사용하는 화면도 함께 오류가 납니다.
- **추천이 없는 사람이 있으면** — API 가 그 고객에게 돌려줄 추천이 없습니다.
- **10줄이 아닌 사람이 있으면** — 고객마다 받는 추천 개수가 달라집니다. 10개를 기대한 화면은 표시가 어긋납니다.
- **상한을 넘는 종목이 있으면** — 안정형 고객에게 위험도 4 \~ 5 종목이 그대로 표시됩니다.
  - 6주차에서 본 적합성원칙에 어긋나는 추천이 고객에게 나가는 것입니다.
- **이미 담은 종목이 있으면** — 고객이 이미 담은 종목을 다시 추천하게 되어 추천의 의미가 없어집니다.
- **한 줄로 정리하면** — 규격 검사는 파일이 API 에 전달되기 전에 형식과 추천 조건을 확인하는 절차입니다.